In [ ]:
#import stuff lol

import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
from scipy.stats.mstats import winsorize
from sklearn.feature_selection import SelectKBest, chi2

In [ ]:
#setup csv variables

pA = pd.read_csv(r'C:/Code/SEC-101/Dido_transactions_A.csv') 
pB = pd.read_csv(r'C:/Code/SEC-101/Dido_transactions_B.csv')

oPA = pA.copy()
oPB = pB.copy()

In [ ]:
#print the first 5 rows of A

pA.head(5)

In [ ]:
#print the first 5 rows of B

pB.head(5)

In [ ]:
#rename columns in pB to match pA for merging

pB.rename(columns={
    "CustomerID": "cust_id", 
    "customer_age": "Age",
    "city_name": "City",
    "amount_spent": "purchase_amount",
    "txn_date": "purchase_date",
    "pay_type": "payment_method",
}, inplace=True)

key = 'cust_id' #the key for merging the two dataframes is the customer ID, which is now named 'cust_id' in both dataframes

pB.head(5)

In [ ]:
#inner merge

iMergedP = pd.merge(pA, pB, how='inner', on=key)
iMergedP.head(10)

#inner merge only keeps the row with complete data in both dataframes, best if you want complete data.
#however, the consequence is omission of some purchases, which is not ideal for tracking all purchases.

In [ ]:
#left merge

lMergedP = pd.merge(pA, pB, how='left', on=key)
lMergedP.head(10)

#left merge only keeps all rows from the left dataframe, but still loses some data from the right dataframe if there is no match in the left dataframe.
#same for right merge, but it keeps all rows from the right dataframe instead of the left dataframe.
#to keep track of all purchases, outer merge is the best option, as it keeps all rows from both dataframes, but may introduce more NaN values for missing data.

In [ ]:
#outer merge

oMergedP = pd.merge(pA, pB, how='outer', on=key)
oMergedP.head(10)

#this is the best option for keeping track of all purchases, as it keeps all rows from both dataframes, but may introduce NaN values for missing data.
#Because overall, it's still better to lose some data than to retain blanks instead of losing an entire row of purchases. Evidence is important.

In [ ]:
#strategies to resolve payment string and age issues

#create a mapping dictionary to standardize payment method strings
pay_map = {"cash": "cash", "credit_card": "credit_card", "credit card": "credit_card", "cc": "credit_card", "paypal": "paypal", "pp": "paypal"}

#apply the mapping to both dataframes, after stripping whitespace and lowercasing the strings
for df in (pA, pB):
    df["payment_method"] = df["payment_method"].str.strip().str.lower().map(pay_map)

#identify unrealistic ages and cut it out
ageList = pd.concat([pA[["cust_id", "Age"]], pB[["cust_id", "Age"]]])
ageList["Age"] = ageList["Age"].where(ageList["Age"].between(18, 122)) #122 is age of oldest person lol

#assign the most common age for each customer ID
age_by_cust = ageList.groupby("cust_id")["Age"].agg(
    lambda s: s.mode().iloc[0] if s.notna().any() else np.nan
)

#fill in the age column in both dataframes with the most common age for each customer ID
for df in (pA, pB):
    df["Age"] = df["cust_id"].map(age_by_cust)


#redefine lMergedP to include the fixed payment_method and Age columns
lMergedP = pd.merge(pA, pB, how="left", on=key, suffixes=("_A", "_B"))
lMergedP.head(10)

#The alternate method I don't use for age issues is stripping out missmatched ages and replacing them with the mean of the remaining ones.
#This is because it definitively introduces innacurracy of the age data. It is better to use the most common age for the best guess as it reduces the risk of incorrect data.

#The alternate method of fixing label mismatch which is less ideal is just lowercasing and stripping the strings
#I chose the full mapping method because addresses abbreviations and misspellings, which the alternative is unable to do.

,cust_id,Age_A,City_A,purchase_amount_A,purchase_date_A,payment_method_A,Age_B,City_B,purchase_amount_B,purchase_date_B,payment_method_B,amount_adjusted
0,1037,74.0,New York,40.69000,2023-12-24,paypal,NaN,NaN,NaN,NaN,NaN,NaN
1,1045,21.0,new york,115.91000,2023-04-30,cash,21.0,New York,NaN,2023-05-18,credit_card,False
2,1015,37.0,PHOENIX,16102.20503,2023-12-09,cash,37.0,phoenix,38.88,2023-10-26,credit_card,False
3,1015,37.0,PHOENIX,16102.20503,2023-12-09,cash,37.0,Phoenix,166.14,2023-01-24,paypal,False
4,1016,35.0,NEW YORK,74.87000,2023-12-25,credit_card,35.0,New York,NaN,2023-04-24,cash,False
5,1016,35.0,NEW YORK,74.87000,2023-12-25,credit_card,35.0,new york,186.07,2023-02-09,credit_card,False
6,1016,35.0,NEW YORK,74.87000,2023-12-25,credit_card,35.0,NaN,146.90,2023-06-05,credit_card,False
7,1021,18.0,LOS ANGELES,13.97000,2023-08-24,cash,18.0,NaN,93.18,2023-06-19,credit_card,True
8,1047,41.0,PHOENIX,425.19000,2023-06-05,cash,41.0,phoenix,13.95,2023-05-17,NaN,False
9,1047,41.0,PHOENIX,425.19000,2023-06-05,cash,41.0,PHOENIX,163.45,2023-12-13,credit_card,False
